> 在线讲义：[第 18 章 · 3D 旋转与刚体变换：动作里的姿态该用什么表示](https://wikieden.github.io/math-and-eval-stats/lessons/18-rotations-rigid-transforms.html) · 本书《数学与实验统计》

# 第 18 章 · 3D 旋转与刚体变换

讲义：[18-rotations-rigid-transforms.md](https://wikieden.github.io/math-and-eval-stats/lessons/18-rotations-rigid-transforms.html)。一格一格往下跑；只算讲义里讲过的例子，练习的数自己算。

## 讲义 ↔ 代码

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 | 旋转矩阵 | `rot_z / rot_x` | (1,0,0) → (0,1,0)；不交换 |
| 第二步 | 万向锁 | `euler_zyx` | pitch 90° 时 yaw、roll 同加 20° 不变 |
| 第三步 | Rodrigues | `rodrigues` | 等于 rot_z(90°) |
| 第四步 | 四元数 | `quat_from_axis_angle / quat_rotate / quat_mul` | (0.7071, 0, 0, 0.7071) |
| 第五步 | 6D | `matrix_from_6d` | RᵀR = I |
| 第六步 | SE(3) | `se3 / se3_inv` | (1, 3, 0) |

## 定义

In [ ]:
import math
import numpy as np

`rot_x`：

In [ ]:
def rot_x(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

`rot_y`：

In [ ]:
def rot_y(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

`rot_z`：

In [ ]:
def rot_z(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

`euler_zyx`：第二步：R = Rz(yaw)·Ry(pitch)·Rx(roll)。

In [ ]:
def euler_zyx(yaw, pitch, roll):
    """第二步：R = Rz(yaw)·Ry(pitch)·Rx(roll)。"""
    return rot_z(yaw) @ rot_y(pitch) @ rot_x(roll)

`skew`：

In [ ]:
def skew(u):
    return np.array([[0, -u[2], u[1]], [u[2], 0, -u[0]], [-u[1], u[0], 0]])

`rodrigues`：第三步：R = I + sin θ·[u]× + (1 - cos θ)·[u]×²。

In [ ]:
def rodrigues(axis, angle):
    """第三步：R = I + sin θ·[u]× + (1 - cos θ)·[u]×²。"""
    u = np.asarray(axis, float) / np.linalg.norm(axis)
    K = skew(u)
    return np.eye(3) + math.sin(angle) * K + (1 - math.cos(angle)) * K @ K

`quat_from_axis_angle`：第四步：q = (cos θ/2, sin θ/2 · u)。

In [ ]:
def quat_from_axis_angle(axis, angle):
    """第四步：q = (cos θ/2, sin θ/2 · u)。"""
    u = np.asarray(axis, float) / np.linalg.norm(axis)
    return np.concatenate([[math.cos(angle / 2)], math.sin(angle / 2) * u])

`quat_mul`：第四步：四元数乘法（Hamilton 约定），q_a ⊗ q_b 表示先转 b 再转 a。

In [ ]:
def quat_mul(a, b):
    """第四步：四元数乘法（Hamilton 约定），q_a ⊗ q_b 表示先转 b 再转 a。"""
    w1, x1, y1, z1 = a
    w2, x2, y2, z2 = b
    return np.array([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2])

`quat_rotate`：第四步：v' = q ⊗ (0, v) ⊗ q*。

In [ ]:
def quat_rotate(q, v):
    """第四步：v' = q ⊗ (0, v) ⊗ q*。"""
    qc = q * np.array([1, -1, -1, -1])
    return quat_mul(quat_mul(q, np.concatenate([[0.0], v])), qc)[1:]

`quat_to_matrix`：

In [ ]:
def quat_to_matrix(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])

`matrix_from_6d`：第五步：6D 表示 → 旋转矩阵（Gram-Schmidt：b1 = a1 归一化，b2 = a2 去掉 b1 分量再归一化，b3 = b1 × b2）。

In [ ]:
def matrix_from_6d(a1, a2):
    """第五步：6D 表示 → 旋转矩阵（Gram-Schmidt：b1 = a1 归一化，b2 = a2 去掉 b1 分量再归一化，b3 = b1 × b2）。"""
    b1 = np.asarray(a1, float) / np.linalg.norm(a1)
    b2 = np.asarray(a2, float) - np.dot(b1, a2) * b1
    b2 /= np.linalg.norm(b2)
    return np.stack([b1, b2, np.cross(b1, b2)], axis=1)

`se3`：第六步：4×4 齐次变换 T = [[R, t], [0, 1]]。

In [ ]:
def se3(R, t):
    """第六步：4×4 齐次变换 T = [[R, t], [0, 1]]。"""
    T = np.eye(4)
    T[:3, :3], T[:3, 3] = R, t
    return T

`se3_inv`：第六步：T⁻¹ = [[Rᵀ, -Rᵀt], [0, 1]]。

In [ ]:
def se3_inv(T):
    """第六步：T⁻¹ = [[Rᵀ, -Rᵀt], [0, 1]]。"""
    R, t = T[:3, :3], T[:3, 3]
    return se3(R.T, -R.T @ t)

## 一步一步跑

### 第一步 旋转矩阵

In [ ]:
D = math.radians; R = rot_z(D(90)); R @ [1, 0, 0], np.allclose(rot_z(D(90)) @ rot_x(D(90)), rot_x(D(90)) @ rot_z(D(90)))

### 第二步 万向锁

In [ ]:
np.allclose(euler_zyx(D(30), D(90), D(10)), euler_zyx(D(50), D(90), D(30)))

### 第三步 Rodrigues

In [ ]:
np.allclose(rodrigues([0, 0, 1], D(90)), R)

### 第四步 四元数

In [ ]:
q = quat_from_axis_angle([0, 0, 1], D(90)); q, quat_rotate(q, np.array([1.0, 0, 0])), np.allclose(quat_to_matrix(q), quat_to_matrix(-q))

### 第五步 6D 表示

In [ ]:
M = matrix_from_6d([1.0, 0.2, 0.0], [0.3, 1.0, 0.1]); np.allclose(M.T @ M, np.eye(3)), np.linalg.det(M)

### 第六步 SE(3)

In [ ]:
T = se3(R, [1, 2, 0]); p = T @ [1, 0, 0, 1]; p[:3], (se3_inv(T) @ p)[:3]

## 代码里才看得到的

1. 本模块四元数是 (w, x, y, z)，scipy 是 (x, y, z, w)。
2. rot_y 的 sin 符号位置和 rot_x、rot_z 相反。
3. 位姿求逆用 [Rᵀ, −Rᵀt]，不用 np.linalg.inv。

## 自己改一改

- pitch 改成 89° 再比万向锁例子
- 验证 q ⊗ q* = (1, 0, 0, 0)

In [ ]:
# 在这里改参数试试


## 口诀

> 三列就是三根轴，转置求逆不交换；
> 中间转到九十度，偏航横滚锁一起；
> 四元半角无万向，q 和负 q 同一转；
> 网络输出用 6D，位姿四乘四串起。